In [4]:

from pathlib import Path
import pandas as pd
import joblib

# Project root and saved files
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "PJME_hourly.csv"
MODEL_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "random_forest_forecaster_v2.joblib"
)

# Load data and trained model
df = pd.read_csv(DATA_PATH, parse_dates=["Datetime"])
model = joblib.load(MODEL_PATH)

# Sort chronologically, preserving the existing training-data rows
df = df.sort_values("Datetime", kind="stable").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["Datetime"].min(), "to", df["Datetime"].max())
print("Model:", type(model).__name__)
print("Model features:", list(model.feature_names_in_))
print("Duplicate timestamps:", df["Datetime"].duplicated().sum())
print("Missing demand values:", df["PJME_MW"].isna().sum())
print("Last 5 rows:")
display(df.tail())


Dataset shape: (145366, 2)
Date range: 2002-01-01 01:00:00 to 2018-08-03 00:00:00
Model: RandomForestRegressor
Model features: ['Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'Lag_1', 'Lag_24', 'Lag_168', 'Rolling_Mean_24', 'Rolling_Mean_168']
Duplicate timestamps: 4
Missing demand values: 0
Last 5 rows:


,Datetime,PJME_MW
145361,2018-08-02 20:00:00,44057.0
145362,2018-08-02 21:00:00,43256.0
145363,2018-08-02 22:00:00,41552.0
145364,2018-08-02 23:00:00,38500.0
145365,2018-08-03 00:00:00,35486.0


In [5]:

# Check the final 200 timestamps for gaps or duplicates
recent = df.tail(200).copy()

time_diffs = recent["Datetime"].diff()

print("Last timestamp:", df["Datetime"].max())
print("\nRecent time-gap distribution:")
print(time_diffs.value_counts().head(10))

print("\nLast 10 timestamp intervals:")
display(
    recent[["Datetime"]]
    .assign(Time_Difference=time_diffs)
    .tail(10)
)


Last timestamp: 2018-08-03 00:00:00

Recent time-gap distribution:
Datetime
0 days 01:00:00    199
Name: count, dtype: int64

Last 10 timestamp intervals:


,Datetime,Time_Difference
145356,2018-08-02 15:00:00,0 days 01:00:00
145357,2018-08-02 16:00:00,0 days 01:00:00
145358,2018-08-02 17:00:00,0 days 01:00:00
145359,2018-08-02 18:00:00,0 days 01:00:00
145360,2018-08-02 19:00:00,0 days 01:00:00
145361,2018-08-02 20:00:00,0 days 01:00:00
145362,2018-08-02 21:00:00,0 days 01:00:00
145363,2018-08-02 22:00:00,0 days 01:00:00
145364,2018-08-02 23:00:00,0 days 01:00:00
145365,2018-08-03 00:00:00,0 days 01:00:00


In [10]:

print("Exact model feature names:")
for name in model.feature_names_in_:
    print(repr(name))


Exact model feature names:
'Hour'
'DayOfWeek'
'Month'
'IsWeekend'
'Lag_1'
'Lag_24'
'Lag_168'
'Rolling_Mean_24'
'Rolling_Mean_168'


In [11]:

import numpy as np
import pandas as pd

# Sort historical data chronologically
history = (
    df[["Datetime", "PJME_MW"]]
    .sort_values("Datetime", kind="stable")
    .reset_index(drop=True)
)

feature_names = list(model.feature_names_in_)
demand_values = history["PJME_MW"].astype(float).tolist()

last_time = history["Datetime"].iloc[-1]
future_times = pd.date_range(
    start=last_time + pd.Timedelta(hours=1),
    periods=24,
    freq="h"
)

forecast_rows = []

for timestamp in future_times:
    pos = len(demand_values)

    features = {
        "Hour": timestamp.hour,
        "DayOfWeek": timestamp.dayofweek,
        "Month": timestamp.month,
        "IsWeekend": int(timestamp.dayofweek >= 5),
        "Lag_1": demand_values[pos - 1],
        "Lag_24": demand_values[pos - 24],
        "Lag_168": demand_values[pos - 168],
        "Rolling_Mean_24": float(np.mean(demand_values[-24:])),
        "Rolling_Mean_168": float(np.mean(demand_values[-168:]))
    }

    X_next = pd.DataFrame(
        [[features[name] for name in feature_names]],
        columns=feature_names
    )

    prediction = float(model.predict(X_next)[0])

    forecast_rows.append({
        "Datetime": timestamp,
        "Forecast_MW": prediction
    })

    # Use prediction as input for the next hour
    demand_values.append(prediction)

future_forecast = pd.DataFrame(forecast_rows)

assert len(future_forecast) == 24
assert future_forecast["Datetime"].is_monotonic_increasing
assert future_forecast["Forecast_MW"].notna().all()

print("Forecast rows:", len(future_forecast))
print("Forecast starts:", future_forecast["Datetime"].min())
print("Forecast ends:", future_forecast["Datetime"].max())
print("Minimum forecast MW:", round(future_forecast["Forecast_MW"].min(), 2))
print("Maximum forecast MW:", round(future_forecast["Forecast_MW"].max(), 2))

display(future_forecast)


Forecast rows: 24
Forecast starts: 2018-08-03 01:00:00
Forecast ends: 2018-08-04 00:00:00
Minimum forecast MW: 28341.76
Maximum forecast MW: 46883.56


,Datetime,Forecast_MW
0,2018-08-03 01:00:00,32462.526178
1,2018-08-03 02:00:00,30414.786791
2,2018-08-03 03:00:00,29046.331990
3,2018-08-03 04:00:00,28341.757776
4,2018-08-03 05:00:00,28431.303781
5,2018-08-03 06:00:00,29847.386309
6,2018-08-03 07:00:00,32205.906116
7,2018-08-03 08:00:00,34695.197798
8,2018-08-03 09:00:00,37051.745461
9,2018-08-03 10:00:00,39083.977704


In [13]:

# Basic validation
print("Number of predictions:", len(future_forecast))
print("Missing predictions:", future_forecast["Forecast_MW"].isna().sum())
print("Duplicate forecast timestamps:", future_forecast["Datetime"].duplicated().sum())

print("\nForecast summary:")
display(future_forecast["Forecast_MW"].describe().to_frame())

# Save forecast output
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

forecast_path = output_dir / "future_24_hour_forecast.csv"
future_forecast.to_csv(forecast_path, index=False)

print("\nSaved to:", forecast_path)


Number of predictions: 24
Missing predictions: 0
Duplicate forecast timestamps: 0

Forecast summary:


,Forecast_MW
count,24.000000
mean,38715.420672
std,6629.950829
min,28341.757776
25%,32398.371162
50%,40158.388525
75%,44415.932568
max,46883.556288



Saved to: c:\Users\ASUS\GridSense\data\processed\future_24_hour_forecast.csv


In [14]:

import plotly.graph_objects as go

# Last 7 days of observed demand
recent_history = history.tail(168)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=recent_history["Datetime"],
    y=recent_history["PJME_MW"],
    mode="lines",
    name="Historical Demand"
))

fig.add_trace(go.Scatter(
    x=future_forecast["Datetime"],
    y=future_forecast["Forecast_MW"],
    mode="lines+markers",
    name="Next 24 Hours Forecast"
))

fig.update_layout(
    title="GridSense: Historical Demand and 24-Hour Forecast",
    xaxis_title="Date and Time",
    yaxis_title="Electricity Demand (MW)",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()


In [16]:

from pathlib import Path
import pandas as pd

# Automatically identify the project root
cwd = Path.cwd()

if cwd.name.lower() == "notebook":
    PROJECT_ROOT = cwd.parent
elif (cwd / "data").exists() and (cwd / "reports").exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = Path(r"C:\Users\ASUS\GridSense")

reports_dir = PROJECT_ROOT / "reports"
processed_dir = PROJECT_ROOT / "data" / "processed"

files_to_check = {
    "Monitoring results": processed_dir / "monitoring_results.csv",
    "Alert history": processed_dir / "alert_history.csv",
    "Forecast test results": processed_dir / "forecast_test_results.csv",
    "Future 24-hour forecast": processed_dir / "future_24_hour_forecast.csv",
}

print("Project root:", PROJECT_ROOT)

for label, path in files_to_check.items():
    print(f"\n--- {label} ---")
    print("Exists:", path.exists())

    if path.exists():
        temp = pd.read_csv(path)
        print("Shape:", temp.shape)
        print("Columns:", temp.columns.tolist())
    else:
        print("File not found.")


Project root: c:\Users\ASUS\GridSense

--- Monitoring results ---
Exists: True
Shape: (29016, 10)
Columns: ['Datetime', 'PJME_MW', 'Forecast', 'Baseline_Prediction', 'Residual', 'Absolute_Error', 'Percentage_Error', 'Rolling_MAE_24h', 'Alert', 'Status']

--- Alert history ---
Exists: True
Shape: (774, 6)
Columns: ['Datetime', 'PJME_MW', 'Forecast', 'Absolute_Error', 'Rolling_MAE_24h', 'Status']

--- Forecast test results ---
Exists: True
Shape: (29073, 4)
Columns: ['Datetime', 'PJME_MW', 'Forecast', 'Baseline_Prediction']

--- Future 24-hour forecast ---
Exists: True
Shape: (24, 2)
Columns: ['Datetime', 'Forecast_MW']


In [17]:

monitoring_path = PROJECT_ROOT / "data" / "processed" / "monitoring_results.csv"

monitoring = pd.read_csv(monitoring_path, parse_dates=["Datetime"])

print("Rows:", len(monitoring))
print("\nMissing values:")
print(monitoring.isna().sum())

print("\nAlert distribution:")
print(monitoring["Alert"].astype(str).value_counts(dropna=False))

print("\nStatus distribution:")
print(monitoring["Status"].astype(str).value_counts(dropna=False))

print("\nRolling MAE summary:")
print(monitoring["Rolling_MAE_24h"].describe())

print("\nFirst and last timestamps:")
print(monitoring["Datetime"].min(), "to", monitoring["Datetime"].max())


Rows: 29016

Missing values:
Datetime               0
PJME_MW                0
Forecast               0
Baseline_Prediction    0
Residual               0
Absolute_Error         0
Percentage_Error       0
Rolling_MAE_24h        0
Alert                  0
Status                 0
dtype: int64

Alert distribution:
Alert
False    28242
True       774
Name: count, dtype: int64

Status distribution:
Status
NORMAL     28242
WARNING      774
Name: count, dtype: int64

Rolling MAE summary:
count    29016.000000
mean       331.390545
std        112.723910
min         89.287483
25%        250.794717
50%        318.687859
75%        396.723523
max       1087.365240
Name: Rolling_MAE_24h, dtype: float64

First and last timestamps:
2015-04-11 22:00:00 to 2018-08-03 00:00:00
